# VisionPort – Inference Baseline Evaluation
## Session 9: Model Performance Benchmarking on IDD (India Driving Dataset)

This notebook establishes the performance baseline for YOLOv8-nano on road-scene images from the India Driving Dataset (IDD).
It records:
- **Speed:** Latency (Mean, Median, P95) and Throughput (images/sec)
- **Resource Utilization:** Process RAM and Peak GPU Memory (allocated & reserved)
- **Inference Detections:** Class labels, confidence scores, and bounding boxes saved to `predictions.json`
- **Summary Report:** Metrics tabulated and exported to `baseline_report.csv`

### Cell 1 – Mount Google Drive
Connects Google Drive to Colab so the notebook can read the model and dataset stored there, and save results back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully!")

### Cell 2 – Install required libraries
Installs Python packages needed for object detection, resource monitoring, and evaluation.

In [ ]:
!pip install -q ultralytics psutil pillow pandas pycocotools

### Cell 3 – Import libraries
Loads every library the notebook needs into memory.

In [ ]:
import os
import json
import time
import zipfile
import statistics
import platform
import torch
import psutil
import pandas as pd
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

print("All libraries imported successfully!")

### Cell 4 – List the top-level files in Google Drive
Verifies that Google Drive is mounted and inspects available files/folders.

In [ ]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")
print("Files in MyDrive:\n")
for item in DRIVE.iterdir():
    print(item.name)

### Cell 5 – Define paths and verify required files exist
Sets all file and folder locations and confirms that both the YOLOv8 model and IDD dataset archive exist.

In [ ]:
DRIVE = Path("/content/drive/MyDrive")
MODEL_PATH = DRIVE / "VisionPort" / "yolov8n.pt"
IDD_ZIP_PATH = DRIVE / "VisionPort" / "IDD_DATASET.zip"
WORK_DIR = Path("/content/VisionPort")
EXTRACT_DIR = WORK_DIR / "IDD"
RESULT_DIR = WORK_DIR / "results"

WORK_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Model exists:", MODEL_PATH.exists())
print("IDD ZIP exists:", IDD_ZIP_PATH.exists())

assert MODEL_PATH.exists(), "YOLO model file not found!"
assert IDD_ZIP_PATH.exists(), "IDD dataset ZIP not found!"
print("\nBoth files found successfully!")

### Cell 6 – Search Drive for model and ZIP files
Troubleshooting utility to locate `.pt` and `.zip` files anywhere under MyDrive.

In [ ]:
print("Searching Google Drive for model and IDD ZIP files...\n")
for file in DRIVE.rglob("*"):
    if file.is_file():
        if file.suffix.lower() == ".pt":
            print("MODEL FOUND:", file)
        elif file.suffix.lower() == ".zip":
            print("ZIP FOUND:", file)

### Cell 7 – Extract the IDD dataset
Extracts the dataset onto the local ephemeral SSD disk (`/content/VisionPort/IDD`) for faster I/O.

In [ ]:
print("Extracting IDD dataset...")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(IDD_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)
print("IDD dataset extracted successfully!")

### Cell 8 – Preview the extracted folder structure
Inspects the directory structure of the extracted IDD dataset.

In [ ]:
print("IDD dataset folders:\n")
for path in list(EXTRACT_DIR.rglob("*"))[:60]:
    print(path.relative_to(EXTRACT_DIR))

### Cell 9 – Find and validate all images
Finds all image files and validates image integrity using PIL.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

all_images = [
    p for p in EXTRACT_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

valid_images = []
invalid_images = []

for image_path in all_images:
    try:
        with Image.open(image_path) as img:
            img.verify()
        valid_images.append(image_path)
    except Exception as e:
        invalid_images.append({
            "file": str(image_path),
            "error": str(e)
        })

print("Total image files:", len(all_images))
print("Valid images:", len(valid_images))
print("Invalid images:", len(invalid_images))

print("\nFirst 10 valid images:")
for image in valid_images[:10]:
    print(image)

### Cell 10 – Select the images for the benchmark
Selects 200 validated images for benchmark consistency.

In [ ]:
if len(valid_images) == 0:
    raise ValueError("No readable images found in the IDD ZIP file.")

benchmark_images = valid_images[:200]
print("\nImages selected for benchmarking:", len(benchmark_images))

### Cell 11 – Check for a GPU
Detects CUDA availability and selects GPU 0 or falls back to CPU.

In [ ]:
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    DEVICE = 0
    print("GPU available:", True)
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("GPU available:", False)
    print("Running benchmark on CPU.")

### Cell 12 – Load the YOLO model
Loads weights from `MODEL_PATH` into Ultralytics YOLO.

In [ ]:
print("Loading YOLO model...")
model = YOLO(str(MODEL_PATH))
print("Model loaded successfully!")

print("\nModel classes:")
for class_id, class_name in model.names.items():
    print(class_id, ":", class_name)

### Cell 13 – Warm up the model
Executes 10 warm-up runs to ensure GPU kernel compilation and memory caches do not bias measurement.

In [ ]:
print("Starting model warm-up...")
WARMUP_RUNS = 10

for i in range(WARMUP_RUNS):
    image_path = str(benchmark_images[i % len(benchmark_images)])
    _ = model.predict(
        source=image_path,
        imgsz=640,
        conf=0.25,
        device=DEVICE,
        verbose=False
    )
    if torch.cuda.is_available():
        torch.cuda.synchronize()

print("Warm-up completed!")

### Cell 14 – The benchmark loop (main measurement)
Executes 3 measurement rounds across 200 images (600 total operations).
Synchronizes CUDA, measures end-to-end latency with `time.perf_counter()`, and tracks memory.

In [ ]:
import numpy as np

IMAGE_SIZE = 640
CONFIDENCE = 0.25
MEASUREMENT_ROUNDS = 3

latencies = []
prediction_records = []

process = psutil.Process(os.getpid())
ram_before = process.memory_info().rss / (1024 ** 2)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

print("\nStarting VisionPort benchmark...\n")
total_start = time.perf_counter()

for round_no in range(MEASUREMENT_ROUNDS):
    print("Measurement round:", round_no + 1)
    for image_path in benchmark_images:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = time.perf_counter()
        results = model.predict(
            source=str(image_path),
            imgsz=IMAGE_SIZE,
            conf=CONFIDENCE,
            device=DEVICE,
            verbose=False
        )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end_time = time.perf_counter()
        latency_ms = (end_time - start_time) * 1000
        latencies.append(latency_ms)
        
        if results is None or len(results) == 0:
            continue
        
        result = results[0]
        detections = []
        if result.boxes is not None:
            for box in result.boxes:
                class_id = int(box.cls[0].item())
                confidence = float(box.conf[0].item())
                xyxy = box.xyxy[0].cpu().tolist()
                detections.append({
                    "class_id": class_id,
                    "class_name": model.names[class_id],
                    "confidence": confidence,
                    "bbox_xyxy": xyxy
                })
        
        if round_no == MEASUREMENT_ROUNDS - 1:
            prediction_records.append({
                "image": str(image_path.relative_to(EXTRACT_DIR)),
                "detections": detections
            })

total_end = time.perf_counter()
total_elapsed = total_end - total_start
ram_after = process.memory_info().rss / (1024 ** 2)

print("\nBenchmark completed!")

### Cell 15 – Calculate and print the performance results
Computes mean, median, P95 latency, overall throughput, process RAM, and peak GPU allocated/reserved memory.

In [ ]:
total_operations = len(latencies)
average_latency = statistics.mean(latencies)
median_latency = statistics.median(latencies)
p95_latency = float(np.percentile(latencies, 95))
throughput = total_operations / total_elapsed
ram_observed = ram_after

print("\n========== VISIONPORT BASELINE RESULTS ==========")
print("Total inference operations:", total_operations)
print("Average latency:", round(average_latency, 2), "ms")
print("Median latency:", round(median_latency, 2), "ms")
print("P95 latency:", round(p95_latency, 2), "ms")
print("Throughput:", round(throughput, 2), "images/sec")
print("Process RAM after benchmark:", round(ram_observed, 2), "MB")

if torch.cuda.is_available():
    gpu_memory = (torch.cuda.max_memory_allocated() / (1024 ** 2))
    gpu_reserved = (torch.cuda.max_memory_reserved() / (1024 ** 2))
    print("GPU peak allocated memory:", round(gpu_memory, 2), "MB")
    print("GPU peak reserved memory:", round(gpu_reserved, 2), "MB")
else:
    gpu_memory = None
    print("GPU memory: N/A - CUDA unavailable")
print("================================================")

### Cell 16 – Save the predictions to a JSON file
Saves all detections from the final benchmark round to `predictions.json`.

In [ ]:
PREDICTION_FILE = RESULT_DIR / "predictions.json"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(
        prediction_records,
        f,
        indent=4
    )

print("Predictions saved:", PREDICTION_FILE)
print("Images with prediction records:", len(prediction_records))

### Cell 17 – Build and save the CSV summary report
Aggregates summary metrics into a pandas DataFrame and exports to `baseline_report.csv`.

In [ ]:
report = {
    "Model": MODEL_PATH.name,
    "Dataset": "IDD",
    "Images": len(benchmark_images),
    "Inference Operations": total_operations,
    "Average Latency (ms)": round(average_latency, 2),
    "Median Latency (ms)": round(median_latency, 2),
    "P95 Latency (ms)": round(p95_latency, 2),
    "Throughput (images/sec)": round(throughput, 2),
    "Process RAM (MB)": round(ram_observed, 2),
    "GPU": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    ),
    "GPU Memory (MB)": (
        round(gpu_memory, 2)
        if gpu_memory is not None
        else "N/A"
    )
}

df = pd.DataFrame([report])
CSV_FILE = RESULT_DIR / "baseline_report.csv"
df.to_csv(CSV_FILE, index=False)
display(df)
print("CSV report saved:", CSV_FILE)

### Cell 18 – Look for annotation files
Checks if annotations exist for calculating mAP or accuracy.

In [ ]:
print("Searching for annotation files...\n")
annotation_files = []
for p in EXTRACT_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in [
        ".json", ".txt", ".yaml", ".yml", ".xml"
    ]:
        annotation_files.append(p)

for p in annotation_files[:100]:
    print(p.relative_to(EXTRACT_DIR))

print("\nTotal annotation/configuration files:", len(annotation_files))

### Cell 19 – Copy results back to Google Drive
Persists `baseline_report.csv` and `predictions.json` to Google Drive.

In [ ]:
DRIVE_RESULTS = DRIVE / "VisionPort_Session9_Results"
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

import shutil
shutil.copy2(
    CSV_FILE,
    DRIVE_RESULTS / "baseline_report.csv"
)
shutil.copy2(
    PREDICTION_FILE,
    DRIVE_RESULTS / "predictions.json"
)

print("Results copied to Google Drive!")
print("Location:", DRIVE_RESULTS)